<a href="https://colab.research.google.com/github/ayushi777lodhi-stack/RAG/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.version.cuda)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
import os

if "COLAB_GPU" in os.environ:
  print("installing req")
  !pip install PyMuPDF
  !pip install tqdm
  !pip install accelerate
  !pip install bitsandbytes
  !pip install flash-attn --no-build-isolation

In [ ]:
!pip uninstall -y torch torchvision transformers sentence-transformers
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -U transformers sentence-transformers

In [ ]:
from google.colab import files

uploaded=files.upload()

pdf_path=list(uploaded.keys())[0]
print("uploaded pdf:",pdf_path)



In [ ]:
import os

print("pdf_path:", pdf_path)
print("Exists:", os.path.exists(pdf_path))

In [ ]:
import fitz

doc = fitz.open(pdf_path)

print("Page count:", doc.page_count)
print("Metadata:", doc.metadata)

In [ ]:
import fitz
from tqdm.auto import tqdm

def text_formatting(text: str)->str:
  cleaned_text=text.replace("\n"," ").strip()
  return cleaned_text


def open_and_read_pdf(pdf_path:str)->list[dict]:
  doc=fitz.open(pdf_path)
  pages_and_texts=[]
  for page_number, page in tqdm(enumerate(doc)):
    text=page.get_text()
    text=text_formatting(text)
    pages_and_texts.append({"page_number":page_number - 41,
                            "page_char_count":len(text),
                            "page_word_count":len(text.split(" ")),
                            "page_sentence_count_raw":len(text.split(". ")),
                            "page_token_count":len(text)/4,
                            "text":text})

  return pages_and_texts

pages_and_texts=open_and_read_pdf(pdf_path=pdf_path)
pages_and_texts[:2]

In [ ]:
print(len(pages_and_texts))

In [ ]:
import random
random.sample(pages_and_texts,k=3)

In [ ]:
from spacy.lang.en import English

nlp=English()

nlp.add_pipe("sentencizer")

In [ ]:
for item in tqdm(pages_and_texts):
  item["sentences"]=list(nlp(item["text"]).sents)

  item["sentences"]=[str(sentence) for sentence in item["sentences"]]

  item["page_sentence_count_spacy"]=len(item["sentences"])

In [ ]:
slice_size=10

def split_list(input_list:list, slice_size:int)->list[list[str]]:
  return [input_list[i:i + slice_size] for i in range(0, len(input_list), slice_size)]

for item in tqdm(pages_and_texts):
  item["sentence_chunks"]=split_list(input_list=item["sentences"], slice_size=slice_size)
  item["num_chunks"]=len(item["sentence_chunks"])

In [ ]:
random.sample(pages_and_texts,k=1)

In [ ]:
import re

pages_and_chunks=[]
for item in tqdm(pages_and_texts):
  for sentence_chunk in item["sentence_chunks"]:
    chunk_dict={}
    chunk_dict["page_number"]=item["page_number"]
    joined_sentence_chunk="".join(sentence_chunk).replace("  "," ").strip()
    joined_sentence_chunk=re.sub(r'\.([A-Z])',r'.\1',joined_sentence_chunk)
    chunk_dict["sentence_chunk"]=joined_sentence_chunk

    chunk_dict["chunk_char_count"]=len(joined_sentence_chunk)
    chunk_dict["chunk_word_count"]=len(joined_sentence_chunk.split(" "))
    chunk_dict["chunk_token_count"]=len(joined_sentence_chunk)/4

    pages_and_chunks.append(chunk_dict)

len(pages_and_chunks)


In [ ]:
random.sample(pages_and_chunks,k=1)

In [ ]:
import pandas as pd
df=pd.DataFrame(pages_and_chunks)
df.describe()

In [ ]:
min_token_length=30
for row in df[df["chunk_token_count"]<=min_token_length].sample(5).iterrows():
  print(f"chunk token count: {row[1]["chunk_token_count"]} | text {row[1]["sentence_chunk"]}")

In [ ]:
pages_and_chunks_over_min_length=df[df["chunk_token_count"]>min_token_length].to_dict(orient="records")
pages_and_chunks_over_min_length[:2]

In [ ]:
from sentence_transformers import util,SentenceTransformer
embedding_model=SentenceTransformer("all-mpnet-base-v2",
                                    device="cuda")
sentences=[
    "The Sentence Transformers library provides an easy and open-source way to create embeddings",
    "Sentences can be embedded one by one or as a list of strings.",
    "Embeddings are one of the most powerful concepts in machine learning",
    "Learn to use embeddings well and you'll be well on your way to being an AI engineer"
]

embeddings=embedding_model.encode(sentences)
embeddings_dict=dict(zip(sentences,embeddings))

for sentence, embedding in embeddings_dict.items():
  print("sentence",sentence)
  print("embedding",embedding)

In [ ]:
text_chunks=[item["sentence_chunk"] for item in pages_and_chunks_over_min_length]

In [ ]:
text_chunk_embeddings=embedding_model.encode(text_chunks, batch_size=32, convert_to_tensor=True)
print(text_chunk_embeddings)

In [ ]:
print(len(text_chunk_embeddings))

In [ ]:
text_chunks_and_embeddings_df=pd.DataFrame(pages_and_chunks_over_min_length)
text_chunks_and_embeddings_df["embedding"] = (text_chunk_embeddings.cpu().numpy().tolist())
embeddings_df_save_path="text_chunks_and_embeddings_df.csv"
text_chunks_and_embeddings_df.to_csv(embeddings_df_save_path,index=False)


In [ ]:
text_chunks_and_embedding_df_load=pd.read_csv(embeddings_df_save_path)
text_chunks_and_embedding_df_load.head()

In [ ]:
df = pd.read_csv("text_chunks_and_embeddings_df.csv")

print(type(df["embedding"].iloc[0]))
print(df["embedding"].iloc[0][:200])

In [ ]:
import ast
import numpy as np

text_chunks_and_embeddings_df = pd.read_csv("text_chunks_and_embeddings_df.csv")
text_chunks_and_embeddings_df["embedding"] = (text_chunks_and_embeddings_df["embedding"].apply(ast.literal_eval).apply(np.array))

pages_and_chunks = text_chunks_and_embeddings_df.to_dict(orient="records")

embeddings = torch.tensor(np.stack(text_chunks_and_embeddings_df["embedding"].values),dtype=torch.float32,device=device)

print(embeddings.shape)

In [ ]:
text_chunks_and_embeddings_df.head()

In [ ]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("all-mpnet-base-v2",device="cuda")
query="macronutrients functions"
print(f"query:{query}")

query_embedding=embedding_model.encode(query, convert_to_tensor=True).to(device)

dot_scores=util.dot_score(query_embedding,embeddings)[0]

top_results_dot_product=torch.topk(dot_scores,k=5)
print(top_results_dot_product)

In [ ]:
import textwrap

def print_wrapped(text,wrap_length=80):
  wrapped_text=textwrap.fill(text,wrap_length)
  print(wrapped_text)

In [ ]:
print(f"query {query}")
print("results:")
for score,idx in zip(top_results_dot_product[0],top_results_dot_product[1]):
  print(f"score:{score:.4f}")
  print("text:")
  print_wrapped(pages_and_chunks[idx]["sentence_chunk"])
  print(f"page number:{pages_and_chunks[idx]["page_number"]}")
  print("\n")

In [ ]:
def retrieve(query,embeddings,n_resources_return,model=embedding_model,):
  query_embedding=model.encode(query,convert_to_tensor=True)
  dot_scores=util.dot_score(query_embedding,embeddings)[0]

  scores,indices=torch.topk(input=dot_scores, k=n_resources_return)

  return scores,indices

In [ ]:
query="symptoms of pellagra"
scores,indices=retrieve(query=query,embeddings=embeddings,n_resources_return=5)
print(scores)
print(f"indices: {indices}")


In [ ]:
from huggingface_hub import login
login(token="")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.utils import is_flash_attn_2_available
from transformers import BitsAndBytesConfig

quantization_config=BitsAndBytesConfig(load_in_4bit=True,
                                       bnb_4bit_compute_dtype=torch.float16)

attn_implementation="sdpa"
model_id = "google/gemma-2-2b-it"
tokenizer=AutoTokenizer.from_pretrained(model_id)
llm_model=AutoModelForCausalLM.from_pretrained(model_id,
                                               torch_dtype=torch.float16,
                                               quantization_config=quantization_config,
                                               low_cpu_mem_usage=False,
                                               attn_implementation=attn_implementation)

use_quantization_config = True
if not use_quantization_config:
  llm_model.to("cuda")

In [ ]:
input_text="what are the macronutrients, and what roles do they play in the human body?"
print(f"input:{input_text}")
prompt_template=[
    {"role":"user",
     "content":input_text}
]

prompt=tokenizer.apply_chat_template(conversation=prompt_template,
                                     tokenize=False,
                                     add_generation_prompt=True)

print(f"prompt formatted :{prompt}")

In [ ]:
input_ids=tokenizer(prompt, return_tensors="pt").to("cuda")
print(f"input ids{input_ids}")
outputs=llm_model.generate(**input_ids,
                           max_new_tokens=256)
print(f"model output :{outputs[0]}\n")

In [ ]:
outputs_decoded=tokenizer.decode(outputs[0])
print(f"decoded model output:{outputs_decoded}\n")

In [ ]:
print(f"input text:{input_text}")
print(f"output text: {outputs_decoded.replace(prompt,'').replace('<bos>','').replace('<eos>','')}")

In [ ]:
questions=[
    "How often should infants be breastfed?",
    "What are the symptoms of pellagra?",
    "What are macronutrients and what roles do they play in the human body?",
    "What is the RDI for protein per day?",
    "How does saliva help with digestion?",
    "water soluble vitamins"
]

query_list=questions

In [ ]:
def prompt_formatter(query,context_items):
  context="- "+"\n-".join([item["sentence_chunk"] for item in context_items])
  base_prompt="""based on the following context items, please answer the query.
  Give yourself room to think by extracting relevant passages from the context before answering the query.
  Make sure the answers are as explantory as possible.
  Use the following examples as refrence for the ideal answer style.
  \n Example 1:
  Query:What are fat-soluble vitamins?
  Answer: The fat-soluble vitamins include Vitamin A, Vitamin D, VItamin E, and VItamin K. These vitamins are absorbed along with the fats in the diet.
  \n Example 2:
  Query: What are the causes of type 2 diabetes?
  Answer: Type 2 diabetes is often associated with overnutrition, particularly the overconsumption of calories leading to obesity.
  \n Example 3:
  Query: What is the importance of hydration for physical performance?
  Answer: Hydration is crucial for physical perfprmance because water plays key roles iin maintaining blood volume, regulationg body temperature, and ensuring better health.
  \nNow use the following context items to answer the use query:
  {context}. The context contains the answer. Look for relevant information.
  \nRelevant passages: <extract relevant passages from the context here>
  User query:{query}
  Answer: """

  base_prompt=base_prompt.format(context=context, query=query)
  prompt_template=[
    {"role":"user",
     "content":input_text}
]

  prompt=tokenizer.apply_chat_template(conversation=prompt_template,
                                     tokenize=False,
                                     add_generation_prompt=True)

  return prompt



In [ ]:
query=random.choice(query_list)
print(f"query:{query}")
scores,indices=retrieve(query=query,embeddings=embeddings,n_resources_return=5)
context_items=[pages_and_chunks[i] for i in indices]
prompt=prompt_formatter(query=query, context_items=context_items)
print(prompt)

In [ ]:
input_ids=tokenizer(prompt, return_tensors="pt").to("cuda")
outputs=llm_model.generate(**input_ids,
                           temperature=0.7,
                           do_sample=True,
                           max_new_tokens=256)

output_text=tokenizer.decode(outputs[0])
print(f"query:{query}")
print(f"answer: \n{output_text.replace(prompt,'')}")